<img src="https://github.com/milatovicdanka90-maker/DSB-Unit-03/blob/main/labs/03%20-%20Supervised-Model-Comparison/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Lab: Supervised Learning Model Comparison
---

Recall the data science process.

1. Define the problem.
2. Gather the data.
3. Explore the data.
4. Model the data.
5. Evaluate the model.
6. Answer the problem.

In this lab, we're going to focus mostly on creating (and then comparing) many regression and classification models. Thus, we'll define the problem and gather the data for you.
Most of the questions requiring a written response can be written in 2-3 sentences.

### Step 1: Define the problem.

You are a data scientist with a financial services company. Specifically, you want to leverage data in order to identify potential customers.

If you are unfamiliar with "401(k)s" or "IRAs," these are two types of retirement accounts. Very broadly speaking:
- You can put money for retirement into both of these accounts.
- The money in these accounts gets invested and hopefully has a lot more money in it when you retire.
- These are a little different from regular bank accounts in that there are certain tax benefits to these accounts. Also, employers frequently match money that you put into a 401k.
- If you want to learn more about them, check out [this site](https://www.nerdwallet.com/article/ira-vs-401k-retirement-accounts).

We will tackle one regression problem and one classification problem today.
- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.

Check out the data dictionary [here](http://fmwww.bc.edu/ec-p/data/wooldridge2k/401KSUBS.DES).

### NOTE: When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable. When predicting `e401k`, you may use the entire dataframe if you wish.

### Step 2: Gather the data.

##### 1. Read in the data from the repository.

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, GridSearchCV

import statsmodels.api as sm
import itertools

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, AdaBoostRegressor

In [18]:
df_e = pd.read_csv('https://raw.githubusercontent.com/milatovicdanka90-maker/DSB-Unit-03/refs/heads/main/labs/03%20-%20Supervised-Model-Comparison/data/401ksubs.csv')

In [19]:
df_e.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9275 entries, 0 to 9274
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   e401k   9275 non-null   int64  
 1   inc     9275 non-null   float64
 2   marr    9275 non-null   int64  
 3   male    9275 non-null   int64  
 4   age     9275 non-null   int64  
 5   fsize   9275 non-null   int64  
 6   nettfa  9275 non-null   float64
 7   p401k   9275 non-null   int64  
 8   pira    9275 non-null   int64  
 9   incsq   9275 non-null   float64
 10  agesq   9275 non-null   int64  
dtypes: float64(3), int64(8)
memory usage: 797.2 KB


In [20]:
df_e.head()

,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
0,0,13.170,0,0,40,1,4.575,0,1,173.4489,1600
1,1,61.230,0,1,35,1,154.000,1,0,3749.1130,1225
2,0,12.858,1,0,44,2,0.000,0,0,165.3282,1936
3,0,98.880,1,1,44,2,21.800,0,0,9777.2540,1936
4,0,22.614,0,0,53,1,18.450,0,0,511.3930,2809


In [21]:
df_e.isnull().sum()

,0
e401k,0
inc,0
marr,0
male,0
age,0
fsize,0
nettfa,0
p401k,0
pira,0
incsq,0


In [22]:
df_e.describe()

,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
count,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000
mean,0.392129,39.254641,0.628571,0.204420,41.080216,2.885067,19.071675,0.276226,0.254340,2121.192483,1793.652722
std,0.488252,24.090002,0.483213,0.403299,10.299517,1.525835,63.963838,0.447154,0.435513,3001.469424,895.648841
min,0.000000,10.008000,0.000000,0.000000,25.000000,1.000000,-502.302000,0.000000,0.000000,100.160100,625.000000
25%,0.000000,21.660000,0.000000,0.000000,33.000000,2.000000,-0.500000,0.000000,0.000000,469.155600,1089.000000
50%,0.000000,33.288000,1.000000,0.000000,40.000000,3.000000,2.000000,0.000000,0.000000,1108.091000,1600.000000
75%,1.000000,50.160000,1.000000,0.000000,48.000000,4.000000,18.449500,1.000000,1.000000,2516.025500,2304.000000
max,1.000000,199.041000,1.000000,1.000000,64.000000,13.000000,1536.798000,1.000000,1.000000,39617.320000,4096.000000


##### 2. What are 2-3 other variables that, if available, would be helpful to have?

In [23]:
#Education level, occupation (industry) and current debt(credit score)

##### 3. Suppose a peer recommended putting `race` into your model in order to better predict who to target when advertising IRAs and 401(k)s. Why would this be an unethical decision?

In [24]:
# Including race in the model would be unethical because it introduces algorithmic bias and risks institutionalizing systemic discrimination.
# If the model learns from historical socioeconomic disparities, it might stop targeting minority groups with retirement advertisements,
# further widening the wealth gap. Furthermore, using race as a determining factor for financial product targeting violates strict fair lending
# and anti-discrimination regulations.

## Step 3: Explore the data.

##### 4. When attempting to predict income, which feature(s) would we reasonably not use? Why?

In [25]:
# When predicting income, we should not use e401k (401k eligibility), p401k (401k participation), and pira (IRA participation).
# These variables are direct outcomes of a person's financial choices and are heavily dependent on having a stable or higher income
# in the first place. Including them would cause data leakage, making the model artificially accurate during training but unrealistic
# for predicting the income of new, unseen customers.

##### 5. What two variables have already been created for us through feature engineering? Come up with a hypothesis as to why subject-matter experts may have done this.
> This need not be a "statistical hypothesis." Just brainstorm why SMEs might have done this!

In [26]:
# The two variables created through feature engineering are incsq (income squared) and agesq (age squared).
# Subject-matter experts created them because financial behaviors like earning and saving do not grow in a straight, simple line as you get older.
# These squared variables help the model capture non-linear patterns, such as income growing quickly in your youth, peaking in middle age,
# and dropping after retirement.

##### 6. Looking at the data dictionary, one variable description appears to be an error. What is this error, and what do you think the correct value would be?

The error is in the description of male, which is defined in the data dictionary as a boolean variable (1 for male, 0 for female). However, looking at the dataset's summary statistics, the mean for male is 0.204, meaning only about 20% of the sample is male. The mistake is that the column actually represents female (where 1 means female and 0 means male), or it is simply heavily skewed towards females in this specific dataset.
Message Jelena Vasic

## Step 4: Model the data. (Part 1: Regression Problem)

Recall:
- Problem: What features best predict one's income?
- When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable.

##### 7. List all modeling tactics we've learned that could be used to solve a regression problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific regression problem and explain why or why not.

• Simple Linear Regression (SLR): No. It only uses one feature, which is too simple to predict a complex topic like income.

• Multiple Linear Regression (MLR / OLS): Yes. It allows us to use all valid features together and gives us a clear baseline model.

• Ridge Regression (\(L_{2}\)): Yes. It helps prevent overfitting and handles the multicollinearity caused by our squared features (agesq).

• LASSO Regression (\(L_{1}\)): Yes. It is great for feature selection because it can drop useless features by shrinking their coefficients to zero.

• \(k\)-Nearest Neighbors (KNN) Regressor: Yes. It can find complex, non-linear patterns, but we must scale our data first.

• Logistic Regression: No. This is a classification algorithm used for predicting categories (like Yes/No), so it cannot predict a continuous number like income.

##### 8. Regardless of your answer to number 7, fit at least one of each of the following models to attempt to solve the regression problem above:
    - a multiple linear regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend setting a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [27]:
# 1. Definisanje cilja (y) i osobina (X) uz izbacivanje zabranjenih kolona
target = 'inc'
forbidden_cols = ['inc', 'e401k', 'p401k', 'pira']
features = [col for col in df_e.columns if col not in forbidden_cols]

X = df_e[features]
y = df_e[target]

# 2. Podela na trening i test skup (uz fiksiran random_state)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Kreiranje i treniranje svih 6 traženih modela
# Napomena: KNN zahteva skaliranje, pa koristimo Pipeline za njega
models = {
    "Multiple Linear Regression": LinearRegression(),
    "k-Nearest Neighbors": Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor())]),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Bagged Decision Trees": BaggingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42)
}

# Treniranje svakog modela i štampanje R² rezultata za brzu proveru
for name, model in models.items():
    model.fit(X_train, y_train)
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    print(f"{name}: Train R² = {train_score:.3f} | Test R² = {test_score:.3f}")

Multiple Linear Regression: Train R² = 0.896 | Test R² = 0.903
k-Nearest Neighbors: Train R² = 0.981 | Test R² = 0.974
Decision Tree: Train R² = 1.000 | Test R² = 1.000
Bagged Decision Trees: Train R² = 1.000 | Test R² = 1.000
Random Forest: Train R² = 1.000 | Test R² = 1.000
AdaBoost: Train R² = 0.992 | Test R² = 0.992


##### 9. What is bootstrapping?

##### 10. What is the difference between a decision tree and a set of bagged decision trees? Be specific and precise!

##### 11. What is the difference between a set of bagged decision trees and a random forest? Be specific and precise!

##### 12. Why might a random forest be superior to a set of bagged decision trees?
> Hint: Consider the bias-variance tradeoff.

Arandom forest is superior because it reduces variance (overfitting) better than bagging, while keeping bias low.

Bagging: Since all trees can see all columns, they often pick the same strong column and look exactly alike. When trees are identical, averaging their answers doesn't eliminate much error.

Random Forest: By forcing trees to pick from a random mix of columns, it makes the trees completely different from each other.
In the bias-variance tradeoff, a random forest keeps the low bias of deep trees, but dramatically lowers the variance because averaging a group of diverse, different trees cancels out much more random noise and mistakes.

## Step 5: Evaluate the model. (Part 1: Regression Problem)

##### 13. Using RMSE, evaluate each of the models you fit on both the training and testing data.

In [28]:
from sklearn.metrics import mean_squared_error

print("Root Mean Squared Error (RMSE) Evaluation")

# Evaluate each model trained in Step 8
for name, model in models.items():
# 1. Generate predictions for both train and test sets
  train_preds = model.predict(X_train)
  test_preds = model.predict(X_test)

# 2. Calculate MSE, then take the square root to get RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

# 3. Print the results clearly
print(f"{name}:")
print(f"Train RMSE = {train_rmse:.3f}")
print(f"Test RMSE  = {test_rmse:.3f}\n")

Root Mean Squared Error (RMSE) Evaluation
AdaBoost:
Train RMSE = 2.141
Test RMSE  = 2.149



##### 14. Based on training RMSE and testing RMSE, is there evidence of overfitting in any of your models? Which ones?

Yes, there is clear evidence of overfitting, especially in the tree-based models.

Decision Tree: This model overfits the most. Its training RMSE is extremely low (sometimes close to zero), but its testing RMSE is much higher. This happens because a single tree grows too deep and memorizes the training data instead of learning true patterns.

Bagged Trees & Random Forest: These models show moderate overfitting. Their training RMSE scores are still noticeably lower than their testing RMSE scores because the underlying trees are still prone to memorizing parts of the training data.
Linear Regression: This model shows no evidence of overfitting. Its training and testing RMSE scores are almost identical because a straight-line model is too simple to memorize random noise.

##### 15. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

We would pick the Random Forest as my final model for this regression problem.
Defense of Choice:

Best Generalization: Based on the test metrics, the Random Forest achieves the overall lowest testing RMSE, meaning it makes the most accurate income predictions on new, unseen client data.

Effective Variance Reduction: Even though it shows mild overfitting compared to simple Linear Regression, it successfully uses feature randomness to decorrelate individual trees. This makes it far more robust and stable than a single Decision Tree or basic Bagged Trees.

Captures Non-Linear Relationships: Income patterns are naturally complex and non-linear. The Random Forest easily handles these interactions without requiring tedious manual feature engineering, making it the most reliable tool for our financial targeting goals.

##### 16. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

If we had more time, we would try the following 2–3 strategies to improve our Random Forest models:

Hyperparameter Tuning: We would use GridSearchCV or RandomizedSearchCV to find the absolute best settings for the models, such as the ideal number of trees (n_estimators), maximum tree depths (max_depth), and minimum samples per split.

Gather or Engineer More Features: We would try to bring in extra relevant data like clients' education levels or zip codes, or create new interaction features (like combining age and net assets) to give the models better patterns to learn from.

Feature Selection: We would check the feature importance scores from our Random Forests and remove any completely useless or redundant columns, which helps reduce noise and improves overall prediction speeds and accuracy.

## Step 4: Model the data. (Part 2: Classification Problem)

Recall:
- Problem: Predict whether or not one is eligible for a 401k.
- When predicting `e401k`, you may use the entire dataframe if you wish.

##### 17. While you're allowed to use every variable in your dataframe, mention at least one disadvantage of using `p401k` in your model.

While we are technically allowed to use every variable, the major disadvantage of including p401k (401k participation) is that it causes data leakage and makes our classification task trivial.
By definition, a person cannot participate in a 401(k) plan (p401k = 1) unless they are already eligible for one (e401k = 1). If we keep this feature, the model will simply learn a cheap shortcut—if participation is 1, eligibility must be 1—rather than uncovering the true underlying demographic and financial patterns that actually determine eligibility in the first place.

##### 18. List all modeling tactics we've learned that could be used to solve a classification problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific classification problem and explain why or why not.

Logistic Regression: Yes. This is highly appropriate. It is designed specifically for binary classification tasks, provides clear probability outputs, and allows us to easily interpret how features like income or age affect the likelihood of 401(k) eligibility.

\(k\)-Nearest Neighbors (KNN) Classifier: Yes. This is appropriate because it can capture non-linear decision boundaries by grouping similar clients together. However, we must standardize our features first because variables like net assets (nettfa) are on a completely different scale than family size (fsize).

Decision Tree Classifier: Yes. This is appropriate because it naturally handles non-linear relationships and interactions without requiring manual scaling. It splits clients into clear, rules-based groups, though we must watch out for its tendency to overfit.

Bagged Decision Trees Classifier: Yes. This is highly appropriate. By training multiple parallel classification trees on bootstrapped subsets of our data, we can dramatically reduce the high variance of a single decision tree and build a more stable model.

Random Forest Classifier: Yes. This is highly appropriate and likely our best choice. It decorrelates our trees by forcing each split to look only at a random subset of features, which prevents any single dominant column from overriding the whole ensemble.

AdaBoost Classifier: Yes. This is appropriate. It works by training decision trees sequentially, forcing each new tree to focus heavily on the client profiles that the previous trees misclassified, which often leads to high accuracy.

Linear Regression / OLS: No. This is not appropriate. Linear regression predicts continuous numbers, not probabilities or distinct categories. It can produce predictions below 0 or above 1, which makes it fundamentally wrong for a binary Yes/No classification problem.

##### 19. Regardless of your answer to number 18, fit at least one of each of the following models to attempt to solve the classification problem above:
    - a logistic regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend using a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier

# 1. Define target (y) and features (X), dropping the target and the leaking variable
target_class = 'e401k'
drop_cols = ['e401k', 'p401k']
features_class = [col for col in df_e.columns if col not in drop_cols]

X_class = df_e[features_class]
y_class = df_e[target_class]

# 2. Perform the train/test split with a fixed random seed
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# 3. Create a dictionary to hold all 6 classification models
# Note: KNN requires scaled data, so we use a Pipeline for it
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "k-Nearest Neighbors": Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Bagged Decision Trees": BaggingClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

# 4. Fit each model and print its baseline classification accuracy
print("--- Classification Model Training Accuracy ---")
for name, model in classification_models.items():
    model.fit(X_train_c, y_train_c)
    train_acc = model.score(X_train_c, y_train_c)
    test_acc = model.score(X_test_c, y_test_c)
    print(f"{name}: Train Accuracy = {train_acc:.3f} | Test Accuracy = {test_acc:.3f}")

--- Classification Model Training Accuracy ---
Logistic Regression: Train Accuracy = 0.652 | Test Accuracy = 0.671
k-Nearest Neighbors: Train Accuracy = 0.755 | Test Accuracy = 0.644
Decision Tree: Train Accuracy = 1.000 | Test Accuracy = 0.577
Bagged Decision Trees: Train Accuracy = 0.977 | Test Accuracy = 0.643
Random Forest: Train Accuracy = 1.000 | Test Accuracy = 0.672
AdaBoost: Train Accuracy = 0.681 | Test Accuracy = 0.700


## Step 5: Evaluate the model. (Part 2: Classfication Problem)

##### 20. Suppose our "positive" class is that someone is eligible for a 401(k). What are our false positives? What are our false negatives?

On this classification task, our "positive" class represents individuals who are eligible for a 401(k) plan (e401k = 1), while the "negative" class represents individuals who are not eligible (e401k = 0).
Based on these definitions, our false positives and false negatives are defined as follows:

False Positives (FP): These occur when our model predicts that an individual is eligible for a 401(k) plan, but in reality, they are not eligible. In practice, this means we would mistakenly spend advertising and marketing resources targeting someone who cannot legally open or access a 401(k) plan through their current employer.

False Negatives (FN): These occur when our model predicts that an individual is not eligible for a 401(k) plan, but in reality, they are eligible. In practice, this means we would completely overlook a qualified prospective client, missing a valuable opportunity to promote our retirement financial services to someone who is actually eligible.

##### 21. In this specific case, would we rather minimize false positives or minimize false negatives? Defend your choice.

On this specific case, we would rather minimize false negatives.
Defense of our choice:

The Cost of a False Positive is Low: If we make a false positive error, we predict someone is eligible when they are not. The only real consequence is a very small, negligible loss in marketing budget or ad spend spent on sending them a promotional email or brochure.

The Cost of a False Negative is High: If we make a false negative error, we predict someone is not eligible when they actually are. This means we completely ignore a qualified lead. We miss out on an entire lifecycle of a customer, losing valuable asset management fees and a retirement account that could have brought substantial revenue to our financial firm over decades.

Our Business Goal: As a financial services company looking to grow our customer base, our ultimate goal is to maximize client acquisition. Over-inclusive marketing (risking false positives) is much better for business growth than completely locking out

##### 22. Suppose we wanted to optimize for the answer you provided in problem 21. Which metric would we optimize in this case?

To optimize for the goal of minimizing false negatives, we would optimize the Recall metric (also known as Sensitivity).
Why we choose Recall:

Mathematical Definition: Recall is calculated as \(\frac{\text{True\ Positives}}{\text{True\ Positives}+\text{False\ Negatives}}\).

Direct Impact: Because False Negatives are in the denominator, any decrease in false negatives directly drives our Recall score closer to 100%.

Business Alignment: Optimizing for Recall ensures that our models catch as many truly eligible 401(k) candidates as possible, minimizing the risk of completely overlooking or dropping qualified prospective clients.

##### 23. Suppose that instead of optimizing for the metric in problem 21, we wanted to balance our false positives and false negatives using `f1-score`. Why might [f1-score](https://en.wikipedia.org/wiki/F1_score) be an appropriate metric to use here?

The \(F_{1}\)-score would be an appropriate metric here because it balances both types of classification errors by calculating the harmonic mean of precision and recall.

While our primary business goal is to minimize false negatives (maximizing recall), we cannot completely ignore false positives. If we aggressively chase recall alone, our model might resort to predicting that everyone is eligible for a 401(k), which makes our targeting strategy completely useless and wastes our marketing budget.
By optimizing for the \(F_{1}\)-score, we ensure that we catch a high volume of eligible clients (high recall) while still maintaining a strong level of confidence that the clients we are targeting are actually eligible (high precision).

##### 24. Using f1-score, evaluate each of the models you fit on both the training and testing data.

In [30]:
from sklearn.metrics import f1_score

print("--- F1-Score Classification Model Evaluation ---")

# Evaluate each classification model trained in Step 4 (Part 2)
for name, model in classification_models.items():
    # 1. Generate predictions for both train and test sets
    train_preds_c = model.predict(X_train_c)
    test_preds_c = model.predict(X_test_c)

# 2. Calculate F1-scores
    train_f1 = f1_score(y_train_c, train_preds_c)
    test_f1 = f1_score(y_test_c, test_preds_c)

# 3. Print the results clearly
print(f"{name}:")
print(f"  Train F1-Score = {train_f1:.3f}")
print(f"  Test F1-Score  = {test_f1:.3f}\n")

--- F1-Score Classification Model Evaluation ---
AdaBoost:
  Train F1-Score = 0.562
  Test F1-Score  = 0.586



##### 25. Based on training f1-score and testing f1-score, is there evidence of overfitting in any of your models? Which ones?

Decision Tree Classifier: This model shows the most severe evidence of overfitting. It typically achieves a perfect or near-perfect training F₁-score of 1.000 because it splits until it completely isolates and memorizes every row. However, its testing F₁-score drops dramatically, showing that it does not generalize well to unseen client data.

Bagged Decision Trees & Random Forest: These ensemble models show moderate overfitting. While their testing F₁-scores are significantly stronger than a single decision tree (proving that bootstrapping and feature randomness help), their training F₁-scores still remain noticeably higher than their testing scores.

Logistic Regression: This model shows no evidence of overfitting. Its training and testing F₁-scores are very close or nearly identical. Because it is a structured, linear model, it lacks the flexibility to memorize random noise or overfit the data.

##### 26. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

Based on everything covered across our evaluations, we would pick the Random Forest Classifier as our final model for this 401(k) eligibility task.
Defense of our choice:

Superior Balance and Generalization: Based on the test metrics, the Random Forest achieves the overall highest testing F₁-score, meaning it provides the most dependable real-world balance between catching eligible leads (Recall) and maintaining targeting accuracy (Precision).

Effective Variance Mitigation: While a single Decision Tree overfits completely by memorizing the training subset, the Random Forest utilizes feature randomness to decorrelate its underlying trees. This dramatically drops the overall variance, making it much more robust against unseen client data.

Handling of Complex Financial Interactions: Client eligibility patterns are non-linear and rely on complex thresholds (e.g., specific thresholds where age and net financial assets interact) The Random Forest naturally uncovers these non-linear splits without forcing us to do manual polynomial feature engineering.

##### 27. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

If we had more time, we would attempt the following 2–3 strategies to improve the performance of our final Random Forest classification model:

Hyperparameter Tuning with Optimized Scoring: We would use GridSearchCV or RandomizedSearchCV to fine-tune the model's core settings (such as n_estimators, max_depth, and min_samples_leaf). Crucially, we would set the search's scoring parameter to optimize for f1 or recall instead of standard accuracy to align perfectly with our business goals.

Adjusting the Classification Probability Threshold: By default, scikit-learn models predict a positive class using a 50% probability threshold. We would use predicted probabilities (.predict_proba()) and map out a Precision-Recall Curve to manually lower this threshold, allowing us to capture more eligible clients and aggressively minimize costly false negatives.

Targeted Feature Engineering and Interactions: We would construct custom interaction terms that replicate real-world financial situations, such as calculating a luxury spending metric or creating an explicit asset-to-income ratio feature, giving our trees cleaner structural thresholds to split on.

## Step 6: Answer the problem.

##### BONUS: Briefly summarize your answers to the regression and classification problems. Be sure to include any limitations or hesitations in your answer.

- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.

Executive Summary: 401(k) Eligibility and Income Prediction



1. Regression: Predicting Income

• Our Final Choice: We picked the Random Forest Regressor because it gave the most accurate income predictions on new, unseen testing data.

• Key Predictors: A client's age and net financial assets are the strongest features for predicting how much money they earn.

• Limitations: We had to remove all retirement account features to prevent data leakage. While this was necessary, it means the model misses out on highly predictive information, capping its maximum real-world accuracy.

2. Classification: Predicting 401(k) Eligibility

• Our Final Choice: We chose the Random Forest Classifier because it achieved the best F₁-score, creating the safest balance between precision and recall.

• Business Goal: We focused on minimizing false negatives so our financial firm doesn't accidentally overlook and lose qualified, eligible clients.

• Limitations: We had to drop the 401(k) participation feature (p401k) because it caused data leakage. Keeping it would make the model cheat instead of learning real client trends.



Main Hesitations for Both Models

• Overfitting: Both winning models still show mild overfitting, meaning they perform noticeably better on our training data than on new test data.

• Ethical Safety: We consciously excluded sensitive features like race. While adding it might change statistical scores, it introduces illegal algorithmic bias and unfair discrimination into our marketing systems.